# Modul 06 · Notebook 03 — RAG di Stack NVIDIA

Di Modul 05 kamu membangun **RAG** (Retrieval-Augmented Generation). Di sini kita jalankan RAG di **stack NVIDIA**: embedding ringan di **CPU**, generator besar di **cloud (NIM)** — jadi tidak ada model berat di GPU-mu.

Ini juga **memperbaiki** notebook lama yang men-generate dengan `GPT-2` (model kecil & usang yang jawabannya kacau). Kita ganti generator-nya dengan **Llama-3.3-70B via NIM**.

> 🔑 Pakai `NVIDIA_API_KEY` yang sama (Colab Secrets) seperti nb02.

In [1]:
# Instal + ambil helper bersama
!pip -q install openai sentence-transformers faiss-cpu
import os
if not os.path.exists("nvidia_utils.py"):
    !wget -q https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/module06-rework/06_nvidia_ecosystem/tools/nvidia_utils.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.9 MB/s eta 0:00:00


In [2]:
# Key dari Colab Secrets
import os
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
print("Key termuat:", bool(os.environ.get("NVIDIA_API_KEY")))

Key termuat: True


In [3]:
# 1) RETRIEVE: korpus kecil -> embedding (multilingual MiniLM, di CPU) -> indeks FAISS
from sentence_transformers import SentenceTransformer
import faiss

emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")

DOCS = [
    "GPU NVIDIA memiliki ribuan core yang bekerja paralel, ideal untuk perkalian matriks pada AI.",
    "NVIDIA NIM menyajikan model besar lewat API kompatibel-OpenAI di integrate.api.nvidia.com.",
    "Quantization 4-bit memangkas memori model secara drastis sehingga model besar muat di GPU kecil.",
    "Tensor Core mempercepat komputasi FP16 di GPU NVIDIA secara signifikan.",
    "RAG menggabungkan pencarian dokumen dengan LLM agar jawaban berdasarkan sumber yang relevan.",
    "CUDA adalah platform NVIDIA untuk memprogram GPU bagi komputasi umum, bukan hanya grafis.",
]

X = emb.encode(DOCS, normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(X.shape[1])     # inner product = cosine (karena sudah dinormalisasi)
index.add(X)
print("Jumlah dokumen terindeks:", index.ntotal)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Jumlah dokumen terindeks: 6


In [4]:
# 2) Cari dokumen paling relevan untuk sebuah pertanyaan
q = "Bagaimana cara menjalankan model besar di GPU yang kecil?"
qv = emb.encode([q], normalize_embeddings=True).astype("float32")
D, I = index.search(qv, k=3)

konteks = "\n".join(f"[{i+1}] {DOCS[i]}" for i in I[0])
print("Konteks terambil:\n", konteks)

Konteks terambil:
 [3] Quantization 4-bit memangkas memori model secara drastis sehingga model besar muat di GPU kecil.
[4] Tensor Core mempercepat komputasi FP16 di GPU NVIDIA secara signifikan.
[1] GPU NVIDIA memiliki ribuan core yang bekerja paralel, ideal untuk perkalian matriks pada AI.


In [5]:
# 3) GENERATE via NIM (BUKAN GPT-2) -- jawaban grounded + sebut nomor sumber
from nvidia_utils import nim_client
client = nim_client()

prompt = (
    "Gunakan HANYA konteks berikut untuk menjawab, dan sebutkan nomor sumber [n].\n"
    f"Konteks:\n{konteks}\n\nPertanyaan: {q}"
)
ans = client.chat.completions.create(
    model="meta/llama-3.3-70b-instruct",
    messages=[{"role": "user", "content": prompt}],
).choices[0].message.content
print(ans)

Cara menjalankan model besar di GPU yang kecil adalah dengan menggunakan Quantization 4-bit yang memangkas memori model secara drastis sehingga model besar muat di GPU kecil [3].


## Yang baru saja terjadi

1. **Retrieve** — embedding ringan (CPU) menemukan dokumen relevan via FAISS.
2. **Augment** — dokumen disisipkan ke prompt sebagai *konteks*.
3. **Generate** — Llama-3.3-70B (NIM) menjawab **berdasarkan konteks** dan menyebut nomor sumber.

Perhatikan: jawaban menyebut **[n]** (sitasi) dan tidak mengarang. Tapi bagaimana kalau model **tetap** mengarang di luar konteks? Itu masalah **halusinasi** — dan di **nb06** kita pasang **grounding rail** (`self check facts`) yang membungkus persis pipeline ini untuk menangkapnya.

## 🧪 Latihan

1. Tambahkan 1 dokumen baru ke `DOCS`, lalu ajukan pertanyaan tentangnya — apakah ikut terambil?
2. Ubah `k` dari 3 menjadi 1 — apakah jawaban jadi kurang lengkap?
3. Ajukan pertanyaan yang **tidak ada** jawabannya di korpus. Apakah model jujur bilang tidak tahu, atau mulai mengarang? (Catat jawabannya — ini motivasi nb06.)